In [93]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial import distance

In [94]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [95]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers=2, num_layers_sleep=2):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(hidden_wake_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        self.sleep_output_size = sleep_output_size

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False):
        # print(x.shape, 'x')
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.sleep_output_size))
            
        # print(x.size())
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw

In [96]:
class compressor(nn.Module):
    def __init__(self, input_size, hidden_compressor_size, num_layers=1):
        super(compressor, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_compressor_size, num_layers, nonlinearity='relu', batch_first=True)
        self.compressor_fc = nn.Linear(hidden_compressor_size, 2)

    def forward(self, x, hc=None):
        if hc == None:
            out, hc = self.rnn(x)
        else:
            out, hc = self.rnn(x, hc)

        out = self.compressor_fc(out)
        
        return out, hc

In [97]:
def compute_geodesic(hidden1, hidden2):

    total_layers = len(hidden1)
    w = 0

    for ii in range(total_layers):
        w_ = np.array(dist( hidden1[ii], hidden2[ii], 'cosine'))
        w += w_
           
    return w[0][0]/total_layers

In [98]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [99]:
class Dataset_converter_compressor(Dataset):
    def __init__(self, data, mask):
        total_sample = len(data)
        self.X = np.zeros((total_sample-2, len(tokens)))
        self.y = np.zeros((total_sample-2, 2))
        for ii in range(total_sample-2):
            token = data[ii]
            self.X[ii, ord(token)-65] = 1 
            self.y[ii,mask[ii]] = 1
            

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [125]:
### initial training ###
total_samples = 30000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 30
hidden_sleep_size = 30
sleep_output_size = 20
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X)
    else:
        predicted_y, hidden = network1(X, hw=mem)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y, y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 1001, loss: 2.0252, accuracy: 0.2440
Iter : 2001, loss: 2.0129, accuracy: 0.2500
Iter : 3001, loss: 1.7397, accuracy: 0.3090
Iter : 4001, loss: 2.1632, accuracy: 0.4300
Iter : 5001, loss: 1.7555, accuracy: 0.5400
Iter : 6001, loss: 2.2840, accuracy: 0.6060
Iter : 7001, loss: 2.6173, accuracy: 0.6180
Iter : 8001, loss: 1.7460, accuracy: 0.6380
Iter : 9001, loss: 1.0814, accuracy: 0.6350
Iter : 10001, loss: 1.5536, accuracy: 0.6330
Iter : 11001, loss: 2.1409, accuracy: 0.6770
Iter : 12001, loss: 2.2877, accuracy: 0.6720
Iter : 13001, loss: 1.9838, accuracy: 0.6690
Iter : 14001, loss: 1.5039, accuracy: 0.6790
Iter : 15001, loss: 2.9234, accuracy: 0.6670
Iter : 16001, loss: 2.1107, accuracy: 0.6720
Iter : 17001, loss: 1.8049, accuracy: 0.6670
Iter : 18001, loss: 1.2288, accuracy: 0.6850
Iter : 19001, loss: 2.1839, accuracy: 0.6630
Iter : 20001, loss: 1.7197, accuracy: 0.6600
Iter : 21001, loss: 1.6624, accuracy: 0.6680
Iter : 22001, loss: 1.5472, accuracy: 0.6700
Iter : 23001, loss:

In [126]:
centroids = []

threshold = .6
n_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0

for ii in range(n_samples):
    if ii == 0:
        # seq += tokens[idx]        
        X_hat, mem = network1(X_hat.reshape(1,1,-1))
        centroids.append(mem[0][0])
    else:
        X_hat, mem = network1(X_hat, mem)

    dis = []
    min_dis = 10
    min_dis_id = -1
    for jj in range(len(centroids)):
        dis.append(
            distance.cosine(centroids[jj].detach().numpy(), mem[0][0].detach().numpy())
        )
        if min_dis >= dis[-1]:
            min_dis = dis[-1] 
            min_dis_id = jj 
    if min_dis < threshold:
        centroids[min_dis_id] = (centroids[min_dis_id] + mem[0][0])/2.0
    else:
        centroids.append(mem[0][0])

    # print(min_dis)   
    X_hat = torch.nn.functional.softmax(X_hat, dim=1)
    dist_categ = torch.distributions.Categorical(probs=X_hat.reshape(-1))
    idx = dist_categ.sample()

    X_hat = torch.zeros(len(tokens),dtype=torch.float32)
    X_hat[idx] = 1.0
    X_hat = X_hat.reshape(1,1,-1)   
    

In [127]:
centroids

[tensor([0.0000, 0.3885, 0.5009, 0.0000, 0.1129, 0.0000, 1.1768, 0.9486, 0.0000,
         0.5741, 0.0802, 0.9473, 0.5492, 0.3907, 0.0519, 0.0000, 0.2220, 0.0000,
         0.1420, 0.0074, 0.2093, 0.2781, 0.6534, 0.0000, 0.0000, 0.0355, 1.2472,
         0.0187, 0.9480, 0.0000], grad_fn=<DivBackward0>),
 tensor([0.0000, 0.3382, 0.0813, 0.0287, 0.0857, 0.0000, 0.5949, 0.3474, 0.0000,
         0.7991, 1.0110, 0.0000, 0.2508, 0.0343, 0.0000, 0.0000, 0.2038, 0.0429,
         0.4496, 0.0083, 0.7760, 0.3306, 0.0000, 0.0000, 0.0000, 0.8721, 0.0929,
         0.3295, 1.6534, 0.1954], grad_fn=<DivBackward0>)]

In [128]:
n_samples = 1000
data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

community = {}
for ii in range(len(centroids)):
    community[ii] = []

ii = 0

for X, y in train_loader:
    if ii == 0:
        # seq += tokens[idx]        
        X_hat, mem = network1(X.reshape(1,1,-1))
        # centroids.append(mem[0][0])
    else:
        X_hat, mem = network1(X, mem)
    
    dis = []
    for center in centroids:
        dis.append(
            distance.cosine(center.detach().numpy(), mem[0][0].detach().numpy())
        )
    
    idx = np.argmin(dis)
    community[idx].append(tokens[X.argmax(axis=2)])

         
    ii += 1

In [129]:
for com in community.keys():
    print(np.unique(community[com]))

['A' 'B' 'C']
['D' 'E' 'F' 'G']


In [90]:
sleep_samples = 40
compressed_seq = ''
data_sleep = get_sequence(sleep_samples, n_community, n_members, train_percent=1.0)
data_set_sleep = Dataset_converter(data_sleep, working_memory, short_term_memory)

sleep_loader = DataLoader(data_set_sleep, batch_size=1, shuffle=False)

network1.rnn.requires_grad = True
network1.wake_fc.requires_grad = True

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
hidden_s = None
correct = np.zeros(1000,dtype=float)
communities = [0]
current_community = 0
# changed_com = False
for X, y in sleep_loader:
    # print( tokens[X.argmax(axis=2)], current_community)
    optimizer.zero_grad()
    # print(data_sleep[total])
    if total == 0:
        predicted_y, hidden_w = network1(X)
    elif total==1:
        predicted_y, hidden_w, hidden_s = network1(X, community, hw=mem, sleep=True)
    else:
        predicted_y, hidden_w, hidden_s = network1(X, community, hw=mem, hs=mem_, sleep=True)
    
    # print(predicted_y.shape)
    loss = criterion(predicted_y, y)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        mem=hidden_w.clone()

        if total != 0:
            mem_=hidden_s.clone()

        if distance.cosine(centroids[0].detach().numpy(), mem[0][0].detach().numpy()) < distance.cosine(centroids[1].detach().numpy(), mem[0][0].detach().numpy()):
             if communities[-1] != 0:
                  communities.append(0)
                  current_community = communities.pop(0)
                  community = centroids[current_community].view(1,1,-1)

             print('community 0', tokens[X.argmax(axis=2)], communities)
        else:
            #  print('yes', communities[-1])
             if communities[-1] != 1:
                  communities.append(1)
                  current_community = communities.pop(0)
                  community = centroids[current_community].view(1,1,-1)
                #   print('eije cuurent', current_community, communities)

             print('community 1', tokens[X.argmax(axis=2)])

        # if total == 0:
        #      community = current_community
        #      prev_community = current_community
        #      prev_community_no = current_community_no
        # else:
        #      if current_community_no != prev_community_no:
        #           community = prev_community
        #           prev_community = current_community
        #           prev_community_no = current_community_no


        true_y = y.argmax(axis=1)
        # print(predicted_y)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

community 1 C
community 1 A
community 1 B
community 1 G
community 0 D [0]
community 0 E [0]
community 0 F [0]
community 1 G
community 0 D [0]
community 0 E [0]
community 0 F [0]
community 1 G
community 1 A
community 1 C
community 1 B
community 1 G
community 0 F [0]
community 0 D [0]
community 0 E [0]
community 1 G
community 0 E [0]
community 0 F [0]
community 0 D [0]
community 1 G
community 0 F [0]
community 0 E [0]
community 0 D [0]
community 1 G
community 0 F [0]
community 0 E [0]
community 0 D [0]
community 1 G
community 0 D [0]
community 0 F [0]
community 0 E [0]
community 1 G
community 0 E [0]
community 0 D [0]
